# 03 — Prepare & Export

Turn the cleaned `countries_clean` panel into the **analysis-ready + sellable** deliverables and
package them (CSV + Excel + Parquet + codebook) into `export/`.

## Two deliverables (why two)

`countries_clean` is a full **annual panel** (216 countries × 1960–2025 × 14 indicators). That's
the right thing to ship as the core dataset, but it's sparse — most cells are empty because WDI
coverage is irregular (Gini especially: only 171 countries have any, and the latest available year
differs per indicator). So we ship two views built from the same clean table:

1. **`countries_panel_v1`** — the full country × year panel, every indicator, with region +
   `gini_welfare_type`. For anyone who wants the time series.
2. **`countries_latest_v1`** — one row per country with the **most recent non-null value per
   indicator**, each carrying the year it came from. This is the "at a glance" table most charts
   and buyers want. Because indicators go stale at different rates, we carry `gini_year` (and
   `gini_welfare_type`) explicitly so the staleness and welfare metric are never hidden.

Neither view alters any value — this stage only reshapes and documents.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection, run_sql, save_processed
from src.prepare import numeric_summary, package_dataset

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Deliverable 1 — the full panel

Read `countries_clean` straight from DuckDB. This is already analysis-ready (country-level, region
attached, welfare flag). We round the float indicators to sensible precision for a clean export and
save an analysis-ready copy to `data/processed/`.

In [ ]:
panel = con.execute('SELECT * FROM countries_clean ORDER BY country_name, year').df()

# round for export readability (values unchanged in meaning)
_round = {
    'gdp_per_capita_ppp': 0, 'gdp_per_capita_nominal': 0, 'population': 0,
    'gdp_growth_pct': 2, 'gini_index': 1, 'life_expectancy': 1, 'fertility_rate': 2,
    'urban_pct': 1, 'unemployment_pct': 2, 'poverty_215_pct': 1, 'poverty_365_pct': 1,
    'inflation_pct': 2, 'trade_pct_gdp': 1, 'internet_pct': 1,
}
for c, nd in _round.items():
    panel[c] = panel[c].round(nd)

save_processed(panel, cfg, 'countries_panel.parquet')
print(f'panel: {len(panel):,} rows × {len(panel.columns)} cols, '
      f'{panel.iso_alpha2.nunique()} countries, {panel.year.min()}-{panel.year.max()}')
panel.head()

## Deliverable 2 — latest snapshot (most recent value per indicator)

For each country, take the **most recent non-null value of each indicator independently** (via a
per-indicator `row_number()` over descending year). Gini additionally carries **`gini_year`** and
**`gini_welfare_type`** — because Gini is the sparsest, stalest series and the metric matters for
any ranking. Every country that exists in the clean panel gets a row (indicators it never reported
stay NULL).

In [ ]:
def _latest(col, con):
    """Most-recent non-null value of `col` per country, as a small DataFrame."""
    return con.execute(f"""
      SELECT iso_alpha2, {col}
      FROM (
        SELECT iso_alpha2, {col},
               row_number() OVER (PARTITION BY iso_alpha2 ORDER BY year DESC) rn
        FROM countries_clean WHERE {col} IS NOT NULL
      ) WHERE rn = 1
    """).df()

# spine: one row per country with its identity/region (region is constant per country)
latest = con.execute("""
  SELECT DISTINCT iso_alpha2, iso_alpha3, country_name, region, sub_region
  FROM countries_clean
""").df()

# Gini carries its year + welfare metric (most-recent Gini observation per country)
gini_latest = con.execute("""
  SELECT iso_alpha2, gini_index, year AS gini_year, gini_welfare_type
  FROM (
    SELECT iso_alpha2, gini_index, year, gini_welfare_type,
           row_number() OVER (PARTITION BY iso_alpha2 ORDER BY year DESC) rn
    FROM countries_clean WHERE gini_index IS NOT NULL
  ) WHERE rn = 1
""").df()
latest = latest.merge(gini_latest, on='iso_alpha2', how='left')

# other indicators: most-recent value each (no per-indicator year column, to keep it readable)
for col in ['gdp_per_capita_ppp', 'gdp_per_capita_nominal', 'gdp_growth_pct',
            'life_expectancy', 'fertility_rate', 'urban_pct', 'unemployment_pct',
            'poverty_215_pct', 'poverty_365_pct', 'inflation_pct', 'trade_pct_gdp',
            'internet_pct', 'population']:
    latest = latest.merge(_latest(col, con), on='iso_alpha2', how='left')

# same rounding as the panel
for c, nd in _round.items():
    if c in latest.columns:
        latest[c] = latest[c].round(nd)

# tidy column order: identity, gini block, then the rest
col_order = (['iso_alpha2', 'iso_alpha3', 'country_name', 'region', 'sub_region',
              'gini_index', 'gini_year', 'gini_welfare_type', 'population',
              'gdp_per_capita_ppp', 'gdp_per_capita_nominal', 'gdp_growth_pct',
              'poverty_215_pct', 'poverty_365_pct', 'life_expectancy', 'fertility_rate',
              'urban_pct', 'unemployment_pct', 'inflation_pct', 'trade_pct_gdp', 'internet_pct'])
latest = latest[col_order].sort_values('country_name').reset_index(drop=True)

save_processed(latest, cfg, 'countries_latest.parquet')
print(f'latest: {len(latest):,} countries × {len(latest.columns)} cols; '
      f'{latest.gini_index.notna().sum()} have a Gini')
latest.head()

## Sanity checks before packaging

Confirm the snapshot is internally consistent and the headline inequality view reads correctly
**with its welfare metric attached** (the whole point of the PIP join).

In [ ]:
# one row per country, no dupes
assert latest['iso_alpha2'].is_unique, 'latest snapshot has duplicate countries!'
# every Gini in the snapshot should carry a year
assert (latest['gini_index'].notna() == latest['gini_year'].notna()).all(), 'gini without a gini_year'
# gini_welfare_type only where gini exists
assert latest.loc[latest.gini_index.isna(), 'gini_welfare_type'].isna().all(), 'welfare flag without a gini'
print('✓ snapshot integrity checks pass')

print('\nTop 10 most unequal (latest Gini), WITH metric + year:')
print(latest.dropna(subset=['gini_index'])
            .nlargest(10, 'gini_index')[['country_name','gini_index','gini_welfare_type','gini_year']]
            .to_string(index=False))

print('\nMost equal 10 (latest Gini):')
print(latest.dropna(subset=['gini_index'])
            .nsmallest(10, 'gini_index')[['country_name','gini_index','gini_welfare_type','gini_year']]
            .to_string(index=False))

numeric_summary(latest)

## Package both deliverables for export

Writes CSV + Excel + Parquet + a plain-English codebook to `export/` for each. The codebook
descriptions below are shared (same columns, minus the panel's `year` vs the snapshot's Gini-year
block). The income-vs-consumption caveat is stated on the Gini columns explicitly.

In [ ]:
CB = {
    'iso_alpha2': 'ISO 3166-1 alpha-2 country code (e.g. US).',
    'iso_alpha3': 'ISO 3166-1 alpha-3 country code (e.g. USA).',
    'country_name': 'World Bank country name.',
    'region': 'UN geoscheme region (Africa, Americas, Asia, Europe, Oceania).',
    'sub_region': 'UN geoscheme sub-region (e.g. Sub-Saharan Africa).',
    'year': 'Observation year (panel only).',
    'gini_index': 'Gini index of income or consumption, 0-100 (World Bank SI.POV.GINI). '
                  'Higher = more unequal. SEE gini_welfare_type: income- and consumption-based '
                  'Ginis are NOT directly comparable (income runs ~4.7 pts higher on average).',
    'gini_year': 'Year the gini_index value is from (snapshot only). Gini is measured in irregular '
                 'survey years, so this often lags the other indicators.',
    'gini_welfare_type': 'Whether the Gini is based on an INCOME or CONSUMPTION welfare aggregate '
                         '(World Bank PIP). NULL if no matching survey. Label/segment rankings by '
                         'this to avoid mixing methods.',
    'population': 'Total population (World Bank SP.POP.TOTL).',
    'gdp_per_capita_ppp': 'GDP per capita, PPP (current international $).',
    'gdp_per_capita_nominal': 'GDP per capita (current US$).',
    'gdp_growth_pct': 'GDP growth, annual %.',
    'poverty_215_pct': 'Poverty headcount ratio at $2.15/day (2017 PPP), % of population.',
    'poverty_365_pct': 'Poverty headcount ratio at $3.65/day (2017 PPP), % of population.',
    'life_expectancy': 'Life expectancy at birth, total years.',
    'fertility_rate': 'Total fertility rate, births per woman.',
    'urban_pct': 'Urban population, % of total.',
    'unemployment_pct': 'Unemployment, % of labor force (ILO modeled estimate).',
    'inflation_pct': 'Inflation, consumer prices, annual %.',
    'trade_pct_gdp': 'Trade (exports + imports) as % of GDP.',
    'internet_pct': 'Individuals using the Internet, % of population.',
}
NOTES = """
Sources: World Bank World Development Indicators (WDI) API v2 and the World Bank Poverty and
Inequality Platform (PIP); ISO 3166-1 + UN geoscheme regions. License: CC-BY 4.0 (World Bank),
public domain (ISO codes). Compiled and cleaned by @unwelcomedata.

Coverage: 216 countries. World Bank AGGREGATES (World, income groups, regions) are excluded.
Namibia and Kosovo are included (handled explicitly during cleaning). Indicator coverage is
irregular — many country-years are blank, especially Gini (only ~171 countries have any).

CRITICAL Gini caveat: gini_index mixes income-based and consumption-based measures across
countries; these are NOT directly comparable (income ~4.7 pts higher on average). Use
gini_welfare_type to filter or label. See SOURCES.md for full methodology and series breaks.
"""

paths_panel = package_dataset(panel, cfg, name='countries_panel_v1',
                              codebook=CB, notes=NOTES)
print()
paths_latest = package_dataset(latest, cfg, name='countries_latest_v1',
                               codebook=CB, notes=NOTES)

## Cleanup

Close the DuckDB connection (single-writer).

In [ ]:
con.close()
print('Connection closed.')

---
**Next:** `04-viz.ipynb` — explore the story (most/least unequal, inequality vs GDP/poverty,
regional patterns), **always filtering or labeling Gini by `gini_welfare_type`**. Settle the
framing there before building any social chart in `06-viz-social`.